In [ ]:
# PharmacoVEP full Colab pipeline
# Research prototype for a pharmacogenetic hackathon.
# Not a medical device and not a prescription.



# Block 1. Install dependencies
Run this cell in Google Colab.



In [ ]:
!pip -q install pandas pydantic jinja2 matplotlib seaborn



# Block 2. File upload and paths
Recommended:
- upload small CSV files directly to `/content`;
- for the large VCF, Google Drive is safer, but direct upload also works if `gzip -t` passes.
Required files:
- `dataset_full.csv`
- `PharmaVEP_final.csv`
- `1kG_Full_dataset.vcf`



In [ ]:
from pathlib import Path
from datetime import datetime, timezone

# If files are in Google Drive, use:
# from google.colab import drive
# drive.mount("/content/drive")
# WORK_DIR = Path("/content/drive/MyDrive/pharmacogenetics/data")
WORK_DIR = Path("/content")

RULES_PATH = WORK_DIR / "dataset_full.csv"
VCF_PATH = WORK_DIR / "1kG_Full_dataset.vcf"
ANNOTATION_CSV_PATH = WORK_DIR / "PharmaVEP_final.csv"

OUT_DIR = WORK_DIR / "pharmavep_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ASSEMBLY = "GRCh38"
RUN_STARTED_AT = datetime.now(timezone.utc).isoformat()

# None = analyze every drug in dataset_full.csv.
# Example subset:
# DRUGS_TO_ANALYZE = ["Warfarin", "Clopidogrel", "Allopurinol"]
DRUGS_TO_ANALYZE = None

print("RULES:", RULES_PATH, RULES_PATH.exists())
print("VCF:", VCF_PATH, VCF_PATH.exists())
print("ANNOTATION:", ANNOTATION_CSV_PATH, ANNOTATION_CSV_PATH.exists())
print("OUT_DIR:", OUT_DIR, OUT_DIR.exists())
print("Run started:", RUN_STARTED_AT)
print("Assembly:", ASSEMBLY)
print("Drugs to analyze:", DRUGS_TO_ANALYZE if DRUGS_TO_ANALYZE else "all drugs from rules")



In [ ]:
# If your files are in Google Drive instead, uncomment and edit this:
#
# from google.colab import drive
# drive.mount("/content/drive")
# WORK_DIR = Path("/content/drive/MyDrive/YOUR_FOLDER")
# RULES_PATH = WORK_DIR / "dataset_full.csv"
# VCF_PATH = WORK_DIR / "1kG_Full_dataset.vcf"
# ANNOTATION_CSV_PATH = WORK_DIR / "PharmaVEP_final.csv"
# OUT_DIR = WORK_DIR / "pharmavep_outputs"
# OUT_DIR.mkdir(parents=True, exist_ok=True)



In [ ]:
# Check that the uploaded files are visible.
!ls -lah /content



In [ ]:
# Check whether the VCF is a valid complete gzip/bgzip file.
# If this prints "unexpected end of file", re-upload the VCF.
!gzip -t "{VCF_PATH}"



# Block 3. Imports and helpers



In [ ]:
import re
import json
import subprocess
from dataclasses import dataclass

import pandas as pd
from pydantic import BaseModel
from jinja2 import Template

RSID_RE = re.compile(r"rs\d+")

def normalize_chrom(chrom):
    chrom = str(chrom).strip()
    return chrom if chrom.startswith("chr") else "chr" + chrom

def make_variant_key(chrom, pos, ref, alt):
    return f"{normalize_chrom(chrom)}:{int(pos)}:{str(ref).upper()}:{str(alt).upper()}"

def extract_rule_rsids(value):
    if pd.isna(value):
        return []
    return RSID_RE.findall(str(value))

def normalize_allele_name(value):
    if pd.isna(value):
        return ""
    text = str(value).strip()
    match = re.search(r"\*[\w]+", text)
    return match.group(0) if match else text



# Block 4. Rule validation



In [ ]:
class RuleModel(BaseModel):
    Drug: str
    Gene: str
    Marker_type: str
    rsID: str | None = None
    Allele: str
    Function: str | None = None
    Phenotype: str | None = None
    Recommendation: str
    CPIC_Level: str | None = None
    Source: str | None = None
    Disease: str | None = None



# Block 5. Load pharmacogenetic rules



In [ ]:
def load_rules(path):
    df = pd.read_csv(path).fillna("")

    required = [
        "Drug", "Gene", "Marker_type", "rsID", "Allele", "Function",
        "Phenotype", "Recommendation", "CPIC_Level", "Source", "Disease"
    ]
    missing = set(required) - set(df.columns)
    if missing:
        raise ValueError(f"dataset_full.csv is missing columns: {sorted(missing)}")

    errors = []
    for idx, row in df.iterrows():
        try:
            RuleModel(**row[required].to_dict())
        except Exception as exc:
            errors.append((idx, exc))

    if errors:
        print("Rule validation errors:")
        for idx, exc in errors[:10]:
            print(idx, exc)
        raise ValueError("dataset_full.csv has validation errors")

    df["rule_id"] = range(1, len(df) + 1)
    df["rsids_norm"] = df["rsID"].apply(extract_rule_rsids)
    df["allele_norm"] = df["Allele"].apply(normalize_allele_name)
    df["marker_type_norm"] = df["Marker_type"].astype(str).str.lower()
    df["is_hla"] = df["Gene"].astype(str).str.upper().eq("HLA-B")

    print("Rules:", len(df))
    print("Drugs:", df["Drug"].nunique())
    print("Genes:", df["Gene"].nunique())
    print("Unique rsIDs:", len(sorted({x for xs in df["rsids_norm"] for x in xs})))
    display(df.head(20))
    return df

rules = load_rules(RULES_PATH)

# Reproducibility: optional drug filtering and exact rules table used in this run.
if DRUGS_TO_ANALYZE is not None:
    rules = rules[rules["Drug"].isin(DRUGS_TO_ANALYZE)].copy()
    missing_drugs = sorted(set(DRUGS_TO_ANALYZE) - set(rules["Drug"].unique()))
    if missing_drugs:
        print("Warning: requested drugs not found in rules:", missing_drugs)

rules_used_path = OUT_DIR / "rules_used.csv"
rules.to_csv(rules_used_path, index=False)

print("Rules used:", len(rules))
print("Drugs used:", sorted(rules["Drug"].unique()))
print("Saved rules_used.csv:", rules_used_path)



# Block 6. Build rsID -> coordinate cache
The VCF stores coordinate IDs, not rsIDs. `PharmaVEP_final.csv` is used only as an annotation table:
`rsID -> CHROM, POS, REF, ALT`.



In [ ]:
def build_rsid_coordinate_cache_from_annotation(annotation_path, rules, cache_path):
    if not annotation_path.exists():
        raise FileNotFoundError(f"Annotation file not found: {annotation_path}")

    ann = pd.read_csv(annotation_path).fillna("")

    required = {"CHROM", "POS", "REF", "ALT"}
    missing = required - set(ann.columns)
    if missing:
        raise ValueError(f"PharmaVEP_final.csv is missing columns: {sorted(missing)}")

    rsid_columns = [c for c in ["rsID", "ID"] if c in ann.columns]
    if not rsid_columns:
        raise ValueError("PharmaVEP_final.csv must have an rsID or ID column")

    wanted_rsids = sorted({rsid for ids in rules["rsids_norm"] for rsid in ids})
    rows = []

    for _, row in ann.iterrows():
        row_rsids = set()
        for col in rsid_columns:
            row_rsids.update(RSID_RE.findall(str(row.get(col, ""))))

        matched = row_rsids.intersection(wanted_rsids)
        if not matched:
            continue

        chrom = normalize_chrom(row["CHROM"])
        pos = int(row["POS"])
        ref = str(row["REF"]).upper()

        for alt in str(row["ALT"]).split(","):
            alt = alt.strip().upper()
            if not alt:
                continue
            key = make_variant_key(chrom, pos, ref, alt)

            for rsid in matched:
                rows.append({
                    "rsid": rsid,
                    "chrom": chrom,
                    "pos": pos,
                    "ref": ref,
                    "alt": alt,
                    "key": key,
                    "status": "resolved",
                    "source": annotation_path.name,
                })

    found = {row["rsid"] for row in rows}
    for rsid in wanted_rsids:
        if rsid not in found:
            rows.append({
                "rsid": rsid,
                "chrom": "",
                "pos": "",
                "ref": "",
                "alt": "",
                "key": "",
                "status": "not_resolved",
                "source": annotation_path.name,
            })

    cache = pd.DataFrame(rows).drop_duplicates()
    cache.to_csv(cache_path, index=False)
    return cache

coord_cache_path = OUT_DIR / "rsid_coordinate_cache.csv"
rsid_coords = build_rsid_coordinate_cache_from_annotation(
    ANNOTATION_CSV_PATH,
    rules,
    coord_cache_path,
)

display(rsid_coords)
print(rsid_coords["status"].value_counts())



# Block 7. Reproducible multi-proxy HLA-B*58:01
This is not true HLA typing. It is a LOW-confidence multi-proxy SNP approximation.
Coordinates are fixed in the notebook for reproducibility and do not depend on Ensembl/network.



In [ ]:
HLA_PROXY_RULES = {
    "HLA-B*58:01": {
        "drug": "Allopurinol",
        "gene": "HLA-B",
        "phenotype": "High risk of severe cutaneous adverse reactions (SCAR)",
        "recommendation": (
            "Обнаружен proxy SNP, связанный с HLA-B*58:01: возможен повышенный риск "
            "тяжелых кожных нежелательных реакций на аллопуринол. "
            "Аллопуринол следует избегать до подтверждения прямым HLA-типированием."
        ),
        "confidence": "LOW",
        "method_note": (
            "HLA-B*58:01 оценен через несколько proxy SNP, а не прямым HLA typing. "
            "Связь proxy SNP с HLA-B*58:01 зависит от популяции."
        ),
        "proxies": [
            {
                "rsid": "rs9263726",
                "ref": "G",
                "risk_allele": "A",
                "chrom": "chr6",
                "pos": 31272120,
                "source_note": "PSORS1C1 rs9263726 G>A",
            },
            {
                "rsid": "rs2734583",
                "ref": "A",
                "risk_allele": "G",
                "chrom": "chr6",
                "pos": 31537703,
                "source_note": "BAT1/DDX39B-region rs2734583 A>G",
            },
            {
                "rsid": "rs3099844",
                "ref": "C",
                "risk_allele": "A",
                "chrom": "chr6",
                "pos": 31481199,
                "source_note": "HCP5-region rs3099844 C>A",
            },
        ],
    }
}

hla_proxy_json_path = OUT_DIR / "hla_proxy_rules.json"
with open(hla_proxy_json_path, "w", encoding="utf-8") as handle:
    json.dump(HLA_PROXY_RULES, handle, ensure_ascii=False, indent=2)

def add_fixed_hla_proxy_coordinates(rsid_coords, hla_proxy_rules, cache_path):
    proxy_rows = []

    for hla_allele, rule in hla_proxy_rules.items():
        for proxy in rule["proxies"]:
            chrom = proxy["chrom"]
            pos = int(proxy["pos"])
            ref = proxy["ref"].upper()
            alt = proxy["risk_allele"].upper()

            proxy_rows.append({
                "rsid": proxy["rsid"],
                "chrom": chrom,
                "pos": pos,
                "ref": ref,
                "alt": alt,
                "key": make_variant_key(chrom, pos, ref, alt),
                "status": "resolved",
                "source": "fixed HLA proxy coordinate table",
            })

    proxy_df = pd.DataFrame(proxy_rows)
    proxy_rsids = set(proxy_df["rsid"])
    rsid_coords = rsid_coords[~rsid_coords["rsid"].isin(proxy_rsids)].copy()

    updated = pd.concat([rsid_coords, proxy_df], ignore_index=True).drop_duplicates()
    updated.to_csv(cache_path, index=False)

    hla_proxy_coords_path = OUT_DIR / "hla_proxy_coordinates.csv"
    proxy_df.to_csv(hla_proxy_coords_path, index=False)

    print("Fixed HLA proxy coordinates:")
    display(proxy_df)
    print("Saved:", hla_proxy_coords_path)

    return updated

rsid_coords = add_fixed_hla_proxy_coordinates(
    rsid_coords=rsid_coords,
    hla_proxy_rules=HLA_PROXY_RULES,
    cache_path=coord_cache_path,
)



# Block 8. Stream-read the large VCF by target coordinates only
This avoids loading the full 1GB VCF into memory.



In [ ]:
@dataclass
class VcfCall:
    sample: str
    chrom: str
    pos: int
    ref: str
    alt: str
    gt: str
    alt_count: int | None
    rsid: str | None = None
    key: str | None = None

def is_gzip_file(path):
    with open(path, "rb") as handle:
        return handle.read(2) == b"\x1f\x8b"

def iter_vcf_lines(path):
    path = Path(path)
    if is_gzip_file(path):
        proc = subprocess.Popen(
            ["gzip", "-dc", str(path)],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            encoding="utf-8",
            errors="replace",
        )
        for line in proc.stdout:
            yield line

        stderr = proc.stderr.read()
        return_code = proc.wait()
        if return_code != 0:
            print("WARNING: gzip exited with code", return_code)
            print(stderr[:1000])
    else:
        with open(path, "rt", encoding="utf-8", errors="replace") as handle:
            for line in handle:
                yield line

def gt_alt_count_for_alt(gt, alt_index):
    if gt is None:
        return None
    alleles = str(gt).replace("|", "/").split("/")
    if "." in alleles:
        return None

    count = 0
    for allele in alleles:
        try:
            if int(allele) == alt_index:
                count += 1
        except ValueError:
            return None
    return count

def parse_sample_gt(sample_field, format_keys):
    values = str(sample_field).split(":")
    data = dict(zip(format_keys, values))
    return data.get("GT", "./.")

def read_vcf_calls_filtered(vcf_path, rsid_coords_df):
    resolved = rsid_coords_df[rsid_coords_df["status"].eq("resolved")].copy()
    key_to_rsids = (
        resolved.groupby("key")["rsid"]
        .apply(lambda x: sorted(set(x)))
        .to_dict()
    )
    target_keys = set(key_to_rsids.keys())

    samples = []
    by_rsid = {}
    by_coord = {}
    variant_rows = []
    total_seen = 0
    total_kept = 0
    malformed_lines = 0

    for line in iter_vcf_lines(vcf_path):
        line = line.rstrip("\n")
        if not line or line.startswith("##"):
            continue

        if line.startswith("#CHROM"):
            header = line.split("\t")
            samples = header[9:]
            by_rsid = {sample: {} for sample in samples}
            by_coord = {sample: {} for sample in samples}
            continue

        if line.startswith("#"):
            continue

        total_seen += 1
        parts = line.split("\t")
        if len(parts) < 10:
            malformed_lines += 1
            continue

        chrom, pos, variant_id, ref, alt_string = parts[0], parts[1], parts[2], parts[3], parts[4]
        chrom = normalize_chrom(chrom)
        try:
            pos = int(pos)
        except ValueError:
            malformed_lines += 1
            continue

        ref = str(ref).upper()
        alts = [x.upper() for x in str(alt_string).split(",")]
        format_keys = parts[8].split(":")
        sample_fields = parts[9:]

        for alt_index, alt in enumerate(alts, start=1):
            key = make_variant_key(chrom, pos, ref, alt)
            if key not in target_keys:
                continue

            total_kept += 1
            rsids = key_to_rsids.get(key, [])
            variant_rows.append({
                "chrom": chrom,
                "pos": pos,
                "id": variant_id,
                "ref": ref,
                "alt": alt,
                "key": key,
                "mapped_rsids": ";".join(rsids),
            })

            for sample, sample_field in zip(samples, sample_fields):
                gt = parse_sample_gt(sample_field, format_keys)
                alt_count = gt_alt_count_for_alt(gt, alt_index)
                call = VcfCall(
                    sample=sample,
                    chrom=chrom,
                    pos=pos,
                    ref=ref,
                    alt=alt,
                    gt=gt,
                    alt_count=alt_count,
                    key=key,
                )
                by_coord[sample][key] = call
                for rsid in rsids:
                    by_rsid[sample][rsid] = call

    variant_index = pd.DataFrame(variant_rows).drop_duplicates()

    print("Samples in VCF:", len(samples))
    print("Variants scanned:", total_seen)
    print("Target variants found:", total_kept)
    print("Unique target coordinates found:", variant_index["key"].nunique() if len(variant_index) else 0)
    print("Malformed lines skipped:", malformed_lines)
    display(variant_index.head(50))

    missing_keys = sorted(target_keys - set(variant_index["key"])) if len(variant_index) else sorted(target_keys)
    if missing_keys:
        print("Missing target coordinates:")
        print(missing_keys[:50])
        print("Total missing:", len(missing_keys))

    return by_rsid, by_coord, variant_index, samples

vcf_by_rsid, vcf_by_coord, variant_index, samples = read_vcf_calls_filtered(
    VCF_PATH,
    rsid_coords,
)

# HLA multi-proxy QA after VCF reading.
hla_proxy_rsids = [
    proxy["rsid"]
    for rule in HLA_PROXY_RULES.values()
    for proxy in rule["proxies"]
]

print("HLA proxy rsIDs:")
print(hla_proxy_rsids)

print("HLA proxy coordinates in rsid_coords:")
display(rsid_coords[rsid_coords["rsid"].isin(hla_proxy_rsids)])

print("HLA proxy variants found in VCF:")
display(
    variant_index[
        variant_index["mapped_rsids"].astype(str).apply(
            lambda value: any(rsid in value for rsid in hla_proxy_rsids)
        )
    ]
)

# Reproducibility: variant-level coverage report.
def build_variant_coverage_report(rules, rsid_coords, variant_index):
    present_keys = set(variant_index["key"].astype(str)) if len(variant_index) else set()
    rows = []

    for _, row in rules.iterrows():
        if row["rsids_norm"]:
            for rsid in row["rsids_norm"]:
                rows.append({
                    "rule_id": row["rule_id"],
                    "drug": row["Drug"],
                    "gene": row["Gene"],
                    "marker_type": row["Marker_type"],
                    "allele": row["Allele"],
                    "rsid": rsid,
                    "source": row["Source"],
                })
        else:
            rows.append({
                "rule_id": row["rule_id"],
                "drug": row["Drug"],
                "gene": row["Gene"],
                "marker_type": row["Marker_type"],
                "allele": row["Allele"],
                "rsid": "",
                "source": row["Source"],
            })

    rules_long = pd.DataFrame(rows)
    coverage = rules_long.merge(
        rsid_coords[["rsid", "chrom", "pos", "ref", "alt", "key", "status"]],
        on="rsid",
        how="left",
    )
    coverage["coordinate_resolved"] = coverage["status"].eq("resolved")
    coverage["present_in_vcf"] = coverage["key"].astype(str).isin(present_keys)
    coverage["coverage_status"] = "not_resolved"
    coverage.loc[
        coverage["coordinate_resolved"] & ~coverage["present_in_vcf"],
        "coverage_status",
    ] = "coordinate_resolved_but_absent_in_vcf"
    coverage.loc[
        coverage["coordinate_resolved"] & coverage["present_in_vcf"],
        "coverage_status",
    ] = "present_in_vcf"
    return coverage

variant_coverage_report = build_variant_coverage_report(rules, rsid_coords, variant_index)
variant_coverage_report_path = OUT_DIR / "variant_coverage_report.csv"
variant_coverage_report.to_csv(variant_coverage_report_path, index=False)

print("Variant coverage report saved:", variant_coverage_report_path)
display(variant_coverage_report.head(30))
display(variant_coverage_report["coverage_status"].value_counts())



# Block 9. Marker interpretation



In [ ]:
STANDARD_NO_PGX_ACTION_RU = (
    "Фармакогенетических оснований для изменения терапии по оцененным маркерам не выявлено; "
    "используйте стандартный режим с обычным клиническим мониторингом."
)

INCOMPLETE_ASSESSMENT_RU = (
    "Оценка неполная: часть клинически значимых маркеров не найдена в доступных данных; "
    "отрицательный результат следует интерпретировать осторожно."
)

REFERENCE_WORDS = {
    "negative", "absent", "wild-type", "wild type", "reference",
    "normal", "no alternate", "no risk marker detected"
}

def is_reference_or_absent_rule(row):
    allele = str(row.get("Allele", "")).lower()
    phenotype = str(row.get("Phenotype", "")).lower()
    function = str(row.get("Function", "")).lower()
    text = " ".join([allele, phenotype, function])
    if row.get("allele_norm") == "*1":
        return True
    return any(word in text for word in REFERENCE_WORDS)

def deduplicate_markers(markers):
    seen = set()
    output = []
    for marker in markers:
        key = (
            marker.get("rsid"),
            marker.get("allele"),
            marker.get("gt"),
            marker.get("key"),
        )
        if key in seen:
            continue
        seen.add(key)
        output.append(marker)
    return output

def genotype_alleles_from_call(call):
    alleles = []
    for part in re.split(r"[\/|]", str(call.gt)):
        if part == ".":
            alleles.append(".")
        elif part == "0":
            alleles.append(call.ref.upper())
        elif part == "1":
            alleles.append(call.alt.upper())
        else:
            alleles.append("?")
    return alleles

def infer_marker_status(sample, group):
    drug = group["Drug"].iloc[0]
    gene = group["Gene"].iloc[0]
    gene_upper = str(gene).upper()
    detected = []
    missing = []
    warnings = []

    if gene_upper == "HLA-B":
        for hla_allele, proxy_rule in HLA_PROXY_RULES.items():
            if str(proxy_rule["drug"]).lower() != str(drug).lower():
                continue

            base_warnings = [
                "HLA оценен через multi-proxy SNP, а не прямым HLA typing.",
                proxy_rule["method_note"],
                "Confidence: LOW.",
            ]

            observed = []
            positive = []
            not_observed = []

            for proxy in proxy_rule["proxies"]:
                proxy_snp = proxy["rsid"]
                risk_allele = proxy["risk_allele"].upper()
                call = vcf_by_rsid.get(sample, {}).get(proxy_snp)

                if call is None:
                    not_observed.append(proxy_snp)
                    continue

                genotype_alleles = genotype_alleles_from_call(call)
                risk_detected = risk_allele in genotype_alleles

                proxy_result = {
                    "rsid": proxy_snp,
                    "risk_allele": risk_allele,
                    "genotype_alleles": genotype_alleles,
                    "genotype_text": "/".join(genotype_alleles),
                    "gt": call.gt,
                    "copies": call.alt_count,
                    "key": call.key,
                    "risk_detected": risk_detected,
                    "source_note": proxy.get("source_note", ""),
                }
                observed.append(proxy_result)
                if risk_detected:
                    positive.append(proxy_result)

            if positive:
                for item in positive:
                    detected.append({
                        "rsid": item["rsid"],
                        "allele": f"{hla_allele} proxy via {item['rsid']}",
                        "allele_norm": hla_allele,
                        "function": "Proxy marker for HLA risk allele",
                        "phenotype": proxy_rule["phenotype"],
                        "recommendation": proxy_rule["recommendation"],
                        "gt": item["gt"],
                        "copies": item["copies"],
                        "key": item["key"],
                        "marker_type": "HLA proxy SNP",
                    })

                positive_text = "; ".join([
                    f"{item['rsid']} {item['genotype_text']}"
                    for item in positive
                ])
                return (
                    f"{hla_allele} proxy-positive ({positive_text})",
                    detected,
                    [],
                    base_warnings + [
                        "Найден хотя бы один proxy SNP с risk allele.",
                        f"Observed proxy SNPs: {[item['rsid'] for item in observed]}",
                        f"Not observed proxy SNPs: {not_observed}",
                    ],
                )

            if observed:
                observed_text = "; ".join([
                    f"{item['rsid']} {item['genotype_text']}"
                    for item in observed
                ])
                return (
                    f"{hla_allele} proxy-negative by observed proxies ({observed_text})",
                    [],
                    [],
                    base_warnings + [
                        "Proxy SNP были найдены в VCF, но risk allele не обнаружен.",
                        f"Observed proxy SNPs: {[item['rsid'] for item in observed]}",
                        f"Not observed proxy SNPs: {not_observed}",
                    ],
                )

            return (
                f"{hla_allele} proxy-not-observed via {', '.join(not_observed)}",
                [],
                [],
                base_warnings + [
                    "Ни один HLA proxy SNP не найден в VCF.",
                    "Для variant-only VCF это может означать отсутствие risk allele, но без gVCF/coverage нельзя отличить истинный reference genotype от отсутствия данных.",
                    "Результат можно трактовать только как proxy-negative с низкой уверенностью.",
                ],
            )

    if gene_upper == "CYP2D6" and group["marker_type_norm"].str.contains("copy number|variation").any():
        warnings.append(
            "CYP2D6 CNV/duplication оценивается неполно: нужен отдельный CYP2D6 CNV/star-allele caller."
        )
    if gene_upper == "CFTR":
        warnings.append(
            "CFTR gating mutations оцениваются панельно; одиночный rsID не покрывает все клинически значимые варианты."
        )
    if gene_upper == "NAT2":
        warnings.append(
            "NAT2 фенотип обычно гаплотипный; текущая оценка упрощена по доступным SNP."
        )
    if gene_upper == "G6PD":
        warnings.append(
            "G6PD X-сцепленный ген; без информации о поле пациента интерпретация упрощена."
        )

    for _, row in group.iterrows():
        if not row["rsids_norm"]:
            if not row["is_hla"]:
                missing.append(str(row["rsID"]))
            continue

        for rsid in row["rsids_norm"]:
            call = vcf_by_rsid.get(sample, {}).get(rsid)
            if call is None or call.alt_count is None:
                missing.append(rsid)
                continue

            if call.alt_count > 0 and not is_reference_or_absent_rule(row):
                detected.append({
                    "rsid": rsid,
                    "allele": row["Allele"],
                    "allele_norm": row["allele_norm"],
                    "function": row["Function"],
                    "phenotype": row["Phenotype"],
                    "recommendation": row["Recommendation"],
                    "gt": call.gt,
                    "copies": call.alt_count,
                    "key": call.key,
                    "marker_type": row["Marker_type"],
                })

    detected = deduplicate_markers(detected)
    missing = sorted(set(missing))
    if detected:
        status = "; ".join([
            f"{m['allele']} ({m['rsid']}, GT={m['gt']})"
            for m in detected
        ])
    else:
        status = "no alternate/risk marker detected"
    return status, detected, missing, warnings



# Block 10. Phenotype mapping



In [ ]:
STAR_ALLELE_RE = re.compile(r"\*[\w]+")

def build_simple_diplotype(gene, detected_markers):
    star_hits = []
    for marker in detected_markers:
        allele = marker.get("allele_norm", "")
        copies = marker.get("copies", 0)
        if not str(allele).startswith("*"):
            continue
        for _ in range(int(copies)):
            star_hits.append(allele)

    if not star_hits:
        return "*1/*1"
    if len(star_hits) == 1:
        return f"*1/{star_hits[0]}"
    return "/".join(star_hits[:2])

def fallback_star_phenotype(gene, diplotype):
    gene = str(gene).upper()
    alleles = STAR_ALLELE_RE.findall(str(diplotype))

    if gene == "CYP2C19":
        if "*17" in alleles and not any(a in {"*2", "*3"} for a in alleles):
            return "Rapid Metabolizer" if alleles.count("*17") == 1 else "Ultrarapid Metabolizer"
        if alleles.count("*2") + alleles.count("*3") == 2:
            return "Poor Metabolizer"
        if any(a in {"*2", "*3"} for a in alleles):
            return "Intermediate Metabolizer"
        return "Normal Metabolizer"

    if gene == "CYP2C9":
        bad = sum(a in {"*2", "*3", "*5", "*6", "*8", "*11"} for a in alleles)
        return ["Normal Metabolizer", "Intermediate Metabolizer", "Poor Metabolizer"][min(bad, 2)]

    if gene == "CYP3A5":
        if alleles == ["*3", "*3"]:
            return "Poor Metabolizer"
        if "*3" in alleles:
            return "Intermediate Metabolizer"
        return "Normal Metabolizer"

    if gene == "CYP2B6":
        bad = sum(a in {"*6", "*18"} for a in alleles)
        return ["Normal Metabolizer", "Intermediate Metabolizer", "Poor Metabolizer"][min(bad, 2)]

    if gene == "TPMT":
        bad = sum(a in {"*2", "*3A", "*3B", "*3C"} for a in alleles)
        return ["Normal Metabolizer", "Intermediate Metabolizer", "Poor Metabolizer"][min(bad, 2)]

    if gene == "CYP2D6":
        bad = sum(a in {"*3", "*4", "*5", "*6"} for a in alleles)
        reduced = sum(a in {"*10", "*17", "*41"} for a in alleles)
        if bad == 2:
            return "Poor Metabolizer"
        if bad == 1 or reduced > 0:
            return "Intermediate Metabolizer"
        return "Normal Metabolizer"

    return "Normal Metabolizer" if diplotype == "*1/*1" else "unknown"

def infer_phenotype(gene, status, detected_markers):
    if status == "not_assessed":
        return "not assessed"

    if detected_markers:
        if any(str(m.get("allele_norm", "")).startswith("*") for m in detected_markers):
            diplotype = build_simple_diplotype(gene, detected_markers)
            return fallback_star_phenotype(gene, diplotype)

        phenotypes = sorted({
            str(m.get("phenotype"))
            for m in detected_markers
            if m.get("phenotype") and str(m.get("phenotype")) != "nan"
        })
        if phenotypes:
            return "; ".join(phenotypes)

    return "normal/reference-like or no risk marker detected"



# Block 11. Recommendations and final table



In [ ]:
def select_recommendation(group, detected_markers, phenotype, missing_markers, status):
    if status == "not_assessed":
        return (
            "Рекомендация не сформирована: генотип/фенотип не удалось надежно оценить "
            "по доступным VCF-данным. " + INCOMPLETE_ASSESSMENT_RU
        )

    if detected_markers:
        recommendations = []
        for marker in detected_markers:
            rec = marker.get("recommendation")
            if rec and str(rec) != "nan":
                recommendations.append(str(rec))
        if recommendations:
            return " ".join(dict.fromkeys(recommendations))

    rec = STANDARD_NO_PGX_ACTION_RU
    if missing_markers:
        rec += " " + INCOMPLETE_ASSESSMENT_RU
    return rec

def classify_actionability(status, detected_markers, missing_markers, warnings):
    if status == "not_assessed":
        return "not_assessed"
    if missing_markers or warnings:
        if detected_markers:
            return "actionable_incomplete"
        return "incomplete"
    if detected_markers:
        return "actionable"
    return "standard"

def interpret_all_samples(rules_df):
    results = []
    grouped = rules_df.groupby(["Drug", "Gene"], sort=True)

    for sample in samples:
        for (drug, gene), group in grouped:
            status, detected, missing, warnings = infer_marker_status(sample, group)
            phenotype = infer_phenotype(gene, status, detected)
            recommendation = select_recommendation(group, detected, phenotype, missing, status)
            actionability = classify_actionability(status, detected, missing, warnings)

            if detected and any(str(m.get("allele_norm", "")).startswith("*") for m in detected):
                diplotype_or_status = build_simple_diplotype(gene, detected)
            else:
                diplotype_or_status = status

            results.append({
                "sample": sample,
                "drug": drug,
                "gene": gene,
                "diplotype_or_status": diplotype_or_status,
                "phenotype": phenotype,
                "recommendation": recommendation,
                "actionability": actionability,
                "detected_markers": "\n".join([
                    f"{m['rsid']} {m['allele']} GT={m['gt']} copies={m['copies']}"
                    for m in detected
                ]) if detected else "—",
                "coverage": f"covered: {len(group) - len(missing)}; missing: {len(missing)}",
                "warnings": "\n".join(warnings) if warnings else "—",
            })

    return pd.DataFrame(results)

result_df = interpret_all_samples(rules)
display(result_df.head(100))

print("Interpretations:", len(result_df))
print("Patients:", result_df["sample"].nunique())
print("Drugs:", result_df["drug"].nunique())
print("Drug-gene pairs:", result_df[["drug", "gene"]].drop_duplicates().shape[0])
display(result_df["actionability"].value_counts())

# Reproducibility: machine-readable run configuration.
run_config = {
    "run_started_at": RUN_STARTED_AT,
    "assembly": ASSEMBLY,
    "work_dir": str(WORK_DIR),
    "rules_path": str(RULES_PATH),
    "vcf_path": str(VCF_PATH),
    "annotation_csv_path": str(ANNOTATION_CSV_PATH),
    "out_dir": str(OUT_DIR),
    "drugs_to_analyze": DRUGS_TO_ANALYZE if DRUGS_TO_ANALYZE is not None else "all",
    "n_rules_used": int(len(rules)),
    "n_drugs": int(rules["Drug"].nunique()),
    "n_genes": int(rules["Gene"].nunique()),
    "n_samples": int(len(samples)),
    "n_target_variants_found": int(len(variant_index)),
    "n_interpretations": int(len(result_df)),
    "outputs": {
        "rules_used": str(rules_used_path),
        "variant_coverage_report": str(variant_coverage_report_path),
        "rsid_coordinate_cache": str(coord_cache_path),
        "hla_proxy_rules": str(hla_proxy_json_path),
    },
    "limitations": [
        "Research prototype; not for clinical decision-making.",
        "HLA-B*58:01 assessed by LOW-confidence proxy SNPs, not direct HLA typing.",
        "CYP2D6 CNV/duplication requires specialized caller.",
        "NAT2 phenotype is simplified and not full haplotype reconstruction.",
        "CFTR gating mutations require panel-level assessment.",
        "G6PD interpretation does not use patient sex.",
    ],
}

run_config_path = OUT_DIR / "run_config.json"
with open(run_config_path, "w", encoding="utf-8") as handle:
    json.dump(run_config, handle, ensure_ascii=False, indent=2)

print("Run config saved:", run_config_path)

# Reproducibility: human-readable methods summary.
methods_text = f"""
# Methods Summary

## Project

This is a research prototype pharmacogenetic interpreter for a hackathon.

**Research prototype; not for clinical decision-making.**

## Inputs

- Rules table: `{RULES_PATH}`
- VCF file: `{VCF_PATH}`
- Annotation table: `{ANNOTATION_CSV_PATH}`
- Genome assembly: `{ASSEMBLY}`

## Rules

The pipeline uses a participant-curated pharmacogenetic rules table.

- Number of rules: {len(rules)}
- Number of drugs: {rules["Drug"].nunique()}
- Number of genes: {rules["Gene"].nunique()}

The exact rules used are saved to:

`{rules_used_path}`

## Variant Matching

The VCF uses coordinate-style variant IDs rather than rsID identifiers. Variants are matched in two steps:

1. Extract relevant rsIDs from the rules table.
2. Convert rsID to `CHROM:POS:REF:ALT` using the annotation table.
3. Stream-read the VCF and extract only target coordinates.

The coordinate cache is saved to:

`{coord_cache_path}`

## HLA-B*58:01

Direct HLA typing is not available from the provided VCF. HLA-B*58:01 is assessed using a LOW-confidence multi-proxy SNP approach.

Proxy SNPs:

- rs9263726
- rs2734583
- rs3099844

This is not equivalent to direct HLA typing. All HLA results require confirmatory HLA typing for clinical use.

## Phenotype Calling

Star-allele based diplotypes are inferred with simplified local logic:

- no alternate allele detected: `*1/*1`
- one alternate star allele: `*1/*X`
- two alternate copies: `*X/*X` or `*X/*Y`

Phenotypes are assigned by local fallback mappings for genes such as CYP2C19, CYP2C9, CYP2B6, CYP2D6, CYP3A5, and TPMT.

## Coverage

Variant coverage is reported in:

`{variant_coverage_report_path}`

Coverage status categories:

- `present_in_vcf`
- `coordinate_resolved_but_absent_in_vcf`
- `not_resolved`

## Known Limitations

- Research prototype; not for clinical decision-making.
- HLA-B*58:01 is proxy-based, not direct HLA typing.
- CYP2D6 CNV/duplication is not reliably inferred from this VCF.
- NAT2 is simplified and does not perform full haplotype phasing.
- CFTR gating mutations require panel-level interpretation.
- G6PD interpretation does not account for patient sex.
- This is not PharmCAT and is not clinically validated.
"""

methods_summary_path = OUT_DIR / "methods_summary.md"
methods_summary_path.write_text(methods_text, encoding="utf-8")

print("Methods summary saved:", methods_summary_path)



# Block 12. Save CSV outputs



In [ ]:
result_csv = OUT_DIR / "personal_pharmacogenetic_report_full.csv"
coverage_csv = OUT_DIR / "variant_coverage_full.csv"

result_df.to_csv(result_csv, index=False)

coverage_df = (
    result_df
    .groupby(["drug", "gene", "actionability"])
    .size()
    .reset_index(name="n")
)
coverage_df.to_csv(coverage_csv, index=False)

print("Saved:")
print(result_csv)
print(coverage_csv)



# Block 13. HTML report



In [ ]:
html_template = Template("""
<!doctype html>
<html>
<head>
<meta charset="utf-8">
<title>Personal pharmacogenetic report</title>
<style>
body {
    background: #111827;
    color: #f9fafb;
    font-family: Arial, sans-serif;
    margin: 32px;
}
h1, h2 { color: #ffffff; }
.sample { margin-bottom: 40px; }
table {
    border-collapse: collapse;
    width: 100%;
    margin-bottom: 24px;
    font-size: 13px;
}
th, td {
    border: 1px solid #374151;
    padding: 8px;
    vertical-align: top;
}
th { background: #1f2937; }
td { background: #111827; }
.actionable { color: #fecaca; font-weight: bold; }
.standard { color: #bbf7d0; }
.incomplete, .actionable_incomplete, .not_assessed { color: #fde68a; }
small { color: #d1d5db; }
</style>
</head>
<body>
<h1>Personal pharmacogenetic report</h1>
<p><small><b>Research prototype; not for clinical decision-making.</b> Not a medical prescription.</small></p>

{% for sample, rows in grouped %}
<div class="sample">
<h2>Sample: {{ sample }}</h2>
<table>
<thead>
<tr>
<th>Drug</th>
<th>Gene</th>
<th>Diplotype/status</th>
<th>Phenotype</th>
<th>Actionability</th>
<th>Recommendation</th>
<th>Detected markers</th>
<th>Coverage / warnings</th>
</tr>
</thead>
<tbody>
{% for r in rows %}
<tr>
<td>{{ r.drug }}</td>
<td>{{ r.gene }}</td>
<td>{{ r.diplotype_or_status }}</td>
<td>{{ r.phenotype }}</td>
<td class="{{ r.actionability }}">{{ r.actionability }}</td>
<td>{{ r.recommendation }}</td>
<td>{{ r.detected_markers | replace('\\n', '<br>') }}</td>
<td>{{ r.coverage }}<br>{{ r.warnings | replace('\\n', '<br>') }}</td>
</tr>
{% endfor %}
</tbody>
</table>
</div>
{% endfor %}
</body>
</html>
""")

grouped = [
    (sample, rows.to_dict(orient="records"))
    for sample, rows in result_df.groupby("sample", sort=True)
]

html = html_template.render(grouped=grouped)
html_path = OUT_DIR / "personal_pharmacogenetic_report_full.html"
html_path.write_text(html, encoding="utf-8")
print("HTML report saved:", html_path)



# Block 14. Plots



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_dir = OUT_DIR / "plots"
plot_dir.mkdir(exist_ok=True)

plt.figure(figsize=(10, 5))
sns.countplot(
    data=result_df,
    x="actionability",
    order=result_df["actionability"].value_counts().index
)
plt.title("Actionability summary")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(plot_dir / "actionability_summary.png", dpi=200)
plt.show()

plt.figure(figsize=(14, 6))
sns.countplot(data=result_df, x="drug", hue="actionability")
plt.title("Actionability by drug")
plt.xticks(rotation=45, ha="right")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(plot_dir / "actionability_by_drug.png", dpi=200)
plt.show()

plt.figure(figsize=(14, 6))
sns.countplot(data=result_df, x="gene", hue="actionability")
plt.title("Actionability by gene")
plt.xticks(rotation=45, ha="right")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(plot_dir / "actionability_by_gene.png", dpi=200)
plt.show()

print("Plots saved to:", plot_dir)

print("Final reproducible outputs:")
for path in [
    rules_used_path,
    coord_cache_path,
    variant_coverage_report_path,
    result_csv,
    coverage_csv,
    html_path,
    run_config_path,
    methods_summary_path,
    hla_proxy_json_path,
]:
    print(path)
